In [ ]:
import sklearn
print(sklearn.__version__)

In [ ]:
import pandas as pd
import boto3
import joblib
import tarfile
import os
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import sagemaker

# 1. Pobranie danych z Twojego S3
bucket = 'cdv-phishing-ak'
file_key = 'phishing.csv'

s3 = boto3.client('s3')
print("Pobieranie pliku z S3...")
s3.download_file(bucket, file_key, 'phishing.csv')

# 2. Wczytanie i NAPRAWA danych
print("Wczytywanie i czyszczenie danych...")
df = pd.read_csv('phishing.csv', on_bad_lines='skip', engine='python')

# Wymuszenie nazw i czyszczenie etykiet z nadmiarowych średników
df.columns = ['URL', 'Label']
df['Label'] = df['Label'].astype(str).apply(lambda x: re.sub(r';+', '', x).strip().lower())

# Mapowanie na wartości numeryczne
df['Label'] = df['Label'].map({'bad': 1, 'good': 0})

# Wyrzucamy rekordy, których nie udało się naprawić
df = df.dropna(subset=['Label', 'URL'])

# Losujemy próbkę do treningu
df = df.sample(n=50000, random_state=42)
print(f"Liczba czystych wierszy do treningu: {len(df)}")

X = df['URL'].astype(str)
y = df['Label']

# 3. Wektoryzacja i Trening (zgodnie z metodologią prowadzącego)
print("Wektoryzacja znakowa i trenowanie modelu...")
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 5), max_features=5000)
X_vec = vectorizer.fit_transform(X)

model = LogisticRegression(max_iter=1000)
model.fit(X_vec, y)

# 4. Pakowanie modelu (Zgodnie ze slajdem 13 instrukcji)
print("Zapisywanie plików do archiwum...")
os.makedirs("model", exist_ok=True)
joblib.dump(model, 'model/model.joblib')
joblib.dump(vectorizer, 'model/vectorizer.joblib')

session = sagemaker.Session()
default_bucket = session.default_bucket()
prefix = "phishing-endpoint"

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("model/model.joblib", arcname="model.joblib")
    tar.add("model/vectorizer.joblib", arcname="vectorizer.joblib")

model_artifact_s3 = session.upload_data("model.tar.gz", bucket=default_bucket, key_prefix=prefix)
print(f"GOTOWE! Model wytrenowany i spakowany: {model_artifact_s3}")

In [ ]:
%%writefile inference.py
import os, json, joblib

def model_fn(model_dir):
    """Ładowanie modelu i wektoryzatora z zapisanego archiwum"""
    model = joblib.load(os.path.join(model_dir, "model.joblib"))
    vectorizer = joblib.load(os.path.join(model_dir, "vectorizer.joblib"))
    return (model, vectorizer)

def input_fn(request_body, request_content_type):
    """Obsługa formatu JSON: {"texts": ["url1", "url2"]}"""
    if request_content_type == "application/json":
        data = json.loads(request_body)
        texts = data.get("texts") or data.get("inputs") or []
        if isinstance(texts, str):
            texts = [texts]
        return texts
    raise ValueError(f"Unsupported content type: {request_content_type}")

def predict_fn(text_list, model_and_vectorizer):
    """Faktyczne rozpoznawanie phishingowych linków z prawdopodobieństwem"""
    model, vectorizer = model_and_vectorizer
    X = vectorizer.transform(text_list)
    preds = model.predict(X)
    
    phishing_probas = None
    if hasattr(model, "predict_proba"):
        # Pobieramy prawdopodobieństwo dla klasy 1 (bad/phishing)
        # model.predict_proba zwraca [p_good, p_bad], więc bierzemy indeks 1
        phishing_probas = model.predict_proba(X)[:, 1]
        
    results = ["phishing" if p == 1 else "bezpieczny" for p in preds]
    
    return {
        "predictions": results, 
        "phishing_score": phishing_probas.tolist() if phishing_probas is not None else None
    }

def output_fn(prediction_dict, accept):
    """Zwrócenie wyniku w formacie JSON"""
    body = json.dumps(prediction_dict)
    return body, "application/json"

In [ ]:
import boto3
from sagemaker import get_execution_role
from sagemaker.sklearn.model import SKLearnModel

role = get_execution_role()
sagemaker_client = boto3.client('sagemaker')
endpoint_name = "phishing-endpoint"

# 1. AUTOMATYCZNE SPRZĄTANIE (zapobiega ValidationException)
try:
    print(f"Sprawdzam i usuwam starą konfigurację: {endpoint_name}...")
    # Usuwamy endpoint i jego konfigurację, jeśli istnieją
    sagemaker_client.delete_endpoint(EndpointName=endpoint_name)
    sagemaker_client.delete_endpoint_config(EndpointConfigName=endpoint_name)
    print("Stara infrastruktura została usunięta.")
except Exception:
    print("Nie znaleziono istniejącej konfiguracji, tworzę nową.")

# 2. KONFIGURACJA MODELU
sk_model = SKLearnModel(
    model_data="s3://sagemaker-us-east-1-590183869608/phishing-endpoint/model.tar.gz",
    role=role,
    framework_version="1.2-1",
    py_version="py3",
    entry_point="inference.py",
    name="phishing-url-model"
)

# 3. WDROŻENIE (DEPLOY)
print("Rozpoczynam wdrożenie serwera (Deploying)...")
predictor = sk_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=endpoint_name
)

print(f"\nSUKCES! Endpoint API jest gotowy do testów: {predictor.endpoint_name}")